In [ ]:
# Primeira tentativa de carregar o modelo deeplabv3plus

import os
import pathlib
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import load_model # type: ignore
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, precision_recall_fscore_support
from dotenv import load_dotenv
import pandas as pd
from datetime import datetime
import time


# Função para exibir tempo decorrido
def format_time(seconds):
    """Formata o tempo em horas, minutos e segundos"""
    m, s = divmod(seconds, 60)
    h, m = divmod(m, 60)
    return f"{int(h):02d}:{int(m):02d}:{int(s):02d}"

# Criar diretório para salvar resultados
RESULTS_DIR = f"resultados_avaliacao_{datetime.now().strftime('%Y-%m-%d_%H-%M-%S')}"
os.makedirs(RESULTS_DIR, exist_ok=True)

# Caminho para o modelo salvo
MODEL_PATH = "deeplabv3plusF_2025-03-27_13-34-11.h5"

# Configurações
NUM_CLASSES = 22
BATCH_SIZE = 8
IMG_SIZE = (512, 512)

print(f"[{datetime.now().strftime('%H:%M:%S')}] Iniciando script de avaliação")
print(f"Modelo: {MODEL_PATH}")
print(f"Resultados serão salvos em: {RESULTS_DIR}")
print(f"Configurações: {NUM_CLASSES} classes, batch size = {BATCH_SIZE}, imagens {IMG_SIZE[0]}x{IMG_SIZE[1]}")

# Definindo a função de perda focal usada no treinamento
def focal_loss(alpha=0.25, gamma=2.0):
    def loss(y_true, y_pred):
        y_pred = tf.keras.backend.clip(y_pred, 1e-7, 1 - 1e-7)
        focal_loss = -alpha * y_true * tf.keras.backend.pow(1 - y_pred, gamma) * tf.keras.backend.log(y_pred)
        focal_loss -= (1 - alpha) * (1 - y_true) * tf.keras.backend.pow(y_pred, gamma) * tf.keras.backend.log(1 - y_pred)
        return tf.keras.backend.mean(focal_loss)
    return loss

# Definindo uma classe para calcular IoU separadamente para cada classe
class MultilabelIoU(tf.keras.metrics.Metric):
    def __init__(self, num_classes=22, threshold=0.5, name='multilabel_iou', **kwargs):
        super(MultilabelIoU, self).__init__(name=name, **kwargs)
        self.num_classes = num_classes
        self.threshold = threshold
        # Manter somas separadas para cada classe
        self.total_intersection = self.add_weight(
            name='total_intersection', 
            shape=(num_classes,),
            initializer='zeros')
        self.total_union = self.add_weight(
            name='total_union', 
            shape=(num_classes,),
            initializer='zeros')
        
    def update_state(self, y_true, y_pred, sample_weight=None):
        # Binarizar previsões
        y_pred = tf.cast(y_pred > self.threshold, tf.float32)
        
        # Calcular intersecção e união para todas as classes
        intersection = tf.reduce_sum(y_true * y_pred, axis=[0, 1, 2])
        union = tf.reduce_sum(y_true, axis=[0, 1, 2]) + tf.reduce_sum(y_pred, axis=[0, 1, 2]) - intersection
        
        # Acumular estatísticas por classe
        self.total_intersection.assign_add(intersection)
        self.total_union.assign_add(union)
        
    def result(self):
        # Calcular IoU para cada classe (evitando divisão por zero)
        iou = tf.where(
            self.total_union > 0, 
            self.total_intersection / (self.total_union + 1e-10),
            tf.zeros_like(self.total_union)
        )
        
        # Calcular média apenas para classes que aparecem nos dados
        valid_classes = tf.cast(self.total_union > 0, tf.float32)
        num_valid_classes = tf.reduce_sum(valid_classes)
        
        mean_iou = tf.cond(
            num_valid_classes > 0,
            lambda: tf.reduce_sum(iou) / num_valid_classes,
            lambda: tf.constant(0.0)
        )
        
        return mean_iou
    
    def reset_state(self):
        self.total_intersection.assign(tf.zeros_like(self.total_intersection))
        self.total_union.assign(tf.zeros_like(self.total_union))
        
    def get_iou_per_class(self):
        # Método para obter o IoU por classe
        return tf.where(
            self.total_union > 0, 
            self.total_intersection / (self.total_union + 1e-10),
            tf.zeros_like(self.total_union)
        )


# Tentativa de carregar o modelo normalmente
def carregar_modelo():
    """Carrega o modelo salvo com métricas personalizadas."""
    print(f"[{datetime.now().strftime('%H:%M:%S')}] Carregando modelo DeepLabV3+...")
    start_time = time.time()
    
    # Definir métricas e funções personalizadas para o carregamento do modelo
    custom_objects = {
        "focal_loss": focal_loss(),
        "MultilabelIoU": MultilabelIoU,
        "mean_io_u": tf.keras.metrics.MeanIoU(num_classes=NUM_CLASSES),
        "precision": tf.keras.metrics.Precision(),
        "recall": tf.keras.metrics.Recall(),
        "accuracy": tf.keras.metrics.Accuracy()
    }
    
    try:
        modelo = load_model(MODEL_PATH, custom_objects=custom_objects)
        elapsed = time.time() - start_time
        print(f"[{datetime.now().strftime('%H:%M:%S')}] Modelo carregado com sucesso! Tempo: {format_time(elapsed)}")
        return modelo
    except Exception as e:
        print(f"[{datetime.now().strftime('%H:%M:%S')}] Erro ao carregar o modelo: {str(e)}")
        try:
            print(f"[{datetime.now().strftime('%H:%M:%S')}] Tentando abordagem alternativa...")
            modelo = load_model(MODEL_PATH, compile=False)
            
            # Compilar o modelo manualmente
            modelo.compile(
                optimizer='adam',  # Não importa para avaliação
                loss=focal_loss(),
                metrics=[
                    'accuracy',
                    tf.keras.metrics.MeanIoU(num_classes=NUM_CLASSES),
                    tf.keras.metrics.Precision(),
                    tf.keras.metrics.Recall()
                ]
            )
            elapsed = time.time() - start_time
            print(f"[{datetime.now().strftime('%H:%M:%S')}] Modelo carregado (segunda tentativa)! Tempo: {format_time(elapsed)}")
            return modelo
        except Exception as e2:
            print(f"[{datetime.now().strftime('%H:%M:%S')}] Falha também na segunda tentativa: {str(e2)}")
            return None


# Tentativa de carregar o modelo desativando a verificação de custom_objects e sem compilação
# def carregar_modelo():
#     import h5py
#     print(f"[{datetime.now().strftime('%H:%M:%S')}] Verificando se o arquivo do modelo existe...")
#     if not os.path.exists(MODEL_PATH):
#         print(f"Arquivo de modelo não encontrado: {MODEL_PATH}")
#         return None
    
#     print(f"[{datetime.now().strftime('%H:%M:%S')}] Tentando carregar com método alternativo...")
#     # Apenas verificar se é um arquivo H5 válido
#     try:
#         with h5py.File(MODEL_PATH, 'r') as f:
#             print(f"Arquivo H5 válido. Grupos principais: {list(f.keys())}")
        
#         # Carregar sem compilar e sem custom_objects
#         modelo = tf.keras.models.load_model(MODEL_PATH, compile=False, custom_objects={})
#         print("Modelo carregado com sucesso!")
#         return modelo
#     except Exception as e:
#         print(f"Erro: {str(e)}")
#         return None


def preparar_dataset_teste():
    """Prepara o dataset de teste"""
    print(f"[{datetime.now().strftime('%H:%M:%S')}] Preparando dataset de teste...")
    start_time = time.time()
    
    # Carregar variáveis de ambiente
    print(f"[{datetime.now().strftime('%H:%M:%S')}] Carregando variáveis de ambiente...")
    load_dotenv()
    
    img_dir_str = os.getenv("BASE_IMG_FOLDER")
    if img_dir_str is None:
        raise ValueError("A variável de ambiente BASE_IMG_FOLDER não está definida no arquivo .env")
    img_dir = pathlib.Path(img_dir_str)
    
    mask_dir_str = os.getenv("NUMPY_FOLDER")
    if mask_dir_str is None:
        raise ValueError("A variável de ambiente NUMPY_FOLDER não está definida no arquivo .env")
    mask_dir = pathlib.Path(mask_dir_str)
    
    # Verificar diretórios
    print(f"[{datetime.now().strftime('%H:%M:%S')}] Verificando diretórios de imagens e máscaras...")
    if not os.path.exists(img_dir):
        raise ValueError(f"Diretório de imagens não encontrado: {img_dir}")
    if not os.path.exists(mask_dir):
        raise ValueError(f"Diretório de máscaras não encontrado: {mask_dir}")
    
    # Listar arquivos
    print(f"[{datetime.now().strftime('%H:%M:%S')}] Listando arquivos de imagens e máscaras...")
    
    # Funções para extrair número do arquivo
    def get_file_number(filepath):
        return int(os.path.basename(filepath).replace('.png', ''))
    
    def get_file_number_mask(filepath):
        return int(os.path.basename(filepath).replace('.npy', ''))
    
    mask_files = [str(path) for path in mask_dir.glob('*.npy') if os.path.exists(path)]
    image_files = [str(path) for path in img_dir.glob('*.png') if os.path.exists(path)]
    
    # Ordenar
    image_files = sorted(image_files, key=get_file_number)
    mask_files = sorted(mask_files, key=get_file_number_mask)
    
    print(f"[{datetime.now().strftime('%H:%M:%S')}] Total de imagens encontradas: {len(image_files)}")
    print(f"[{datetime.now().strftime('%H:%M:%S')}] Total de máscaras encontradas: {len(mask_files)}")
    
    # Verificar se os números correspondem
    if len(image_files) != len(mask_files):
        print(f"[{datetime.now().strftime('%H:%M:%S')}] AVISO: Número de imagens ({len(image_files)}) é diferente do número de máscaras ({len(mask_files)})!")
    
    # Função para carregar e pré-processar dados
    def load_and_preprocess_image_mask(image_path, mask_path):
        try:
            # Carregar imagem RGB
            img = tf.io.read_file(image_path)
            img = tf.image.decode_png(img, channels=3)
            img = tf.image.resize(img, IMG_SIZE)
            img = tf.cast(img, tf.float32) / 255.0

            # Carregar máscara do arquivo numpy
            mask = np.load(mask_path.numpy().decode())
            mask = tf.convert_to_tensor(mask, dtype=tf.float32)

            # Normalizar máscara
            mask = mask / 255.0
            mask = tf.where(mask >= 0.3, 1.0, 0.0)

            return img, mask

        except Exception as e:
            print(f"Erro ao processar imagem/máscara {image_path}, {mask_path}: {str(e)}")
            raise

    def process_path(image_path, mask_path):
        img, mask = tf.py_function(load_and_preprocess_image_mask, [image_path, mask_path], [tf.float32, tf.float32])
        img.set_shape(IMG_SIZE + (3,))
        mask.set_shape(IMG_SIZE + (NUM_CLASSES,))
        return img, mask
    
    # Criar dataset
    print(f"[{datetime.now().strftime('%H:%M:%S')}] Criando dataset TensorFlow...")
    dataset = tf.data.Dataset.from_tensor_slices((image_files, mask_files))
    dataset = dataset.map(process_path, num_parallel_calls=tf.data.AUTOTUNE)
    
    # Calcular cardinalidade
    dataset_size = tf.data.experimental.cardinality(dataset).numpy()
    print(f"[{datetime.now().strftime('%H:%M:%S')}] Tamanho total do dataset: {dataset_size} imagens")
    
    # Dividir dataset
    train_size = int(0.8 * dataset_size)
    val_size = int(0.1 * dataset_size)
    test_size = dataset_size - train_size - val_size
    
    print(f"[{datetime.now().strftime('%H:%M:%S')}] Dividindo dataset:")
    print(f"  - Treino (80%): {train_size} imagens")
    print(f"  - Validação (10%): {val_size} imagens")
    print(f"  - Teste (10%): {test_size} imagens")
    
    # Aplicar shuffle com seed para reprodutibilidade
    shuffled_dataset = dataset.shuffle(buffer_size=32, seed=42)
    
    # Dividir o dataset
    train_dataset = shuffled_dataset.take(train_size)
    remaining_dataset = shuffled_dataset.skip(train_size)
    val_dataset = remaining_dataset.take(val_size)
    test_dataset = remaining_dataset.skip(val_size)
    
    # Verificar cardinalidade dos datasets
    train_card = tf.data.experimental.cardinality(train_dataset).numpy()
    val_card = tf.data.experimental.cardinality(val_dataset).numpy()
    test_card = tf.data.experimental.cardinality(test_dataset).numpy()
    
    print(f"[{datetime.now().strftime('%H:%M:%S')}] Cardinalidade dos datasets:")
    print(f"  - Treino: {train_card if train_card >= 0 else 'Desconhecido'}")
    print(f"  - Validação: {val_card if val_card >= 0 else 'Desconhecido'}")
    print(f"  - Teste: {test_card if test_card >= 0 else 'Desconhecido'}")
    
    # Configurar batch para o dataset de teste
    print(f"[{datetime.now().strftime('%H:%M:%S')}] Aplicando batch ({BATCH_SIZE}) ao dataset de teste...")
    test_dataset = test_dataset.batch(BATCH_SIZE).prefetch(1)
    
    elapsed = time.time() - start_time
    print(f"[{datetime.now().strftime('%H:%M:%S')}] Dataset de teste preparado! Tempo: {format_time(elapsed)}")
    
    return test_dataset

def avaliar_modelo():
    """Função principal para avaliar o modelo e gerar relatórios."""
    total_start_time = time.time()
    print(f"[{datetime.now().strftime('%H:%M:%S')}] Iniciando avaliação completa do modelo")
    
    # Nomes das classes (adaptar conforme necessário)
    CLASS_NAMES = [f"Classe_{i+1}" for i in range(NUM_CLASSES)]
    
    # Carregar o modelo
    modelo = carregar_modelo()
    if modelo is None:
        print(f"[{datetime.now().strftime('%H:%M:%S')}] Falha crítica ao carregar o modelo. Encerrando avaliação.")
        return
    
    # Preparar dataset de teste
    test_dataset = preparar_dataset_teste()
    
    # Avaliação usando model.evaluate()
    print(f"\n[{datetime.now().strftime('%H:%M:%S')}] Iniciando avaliação com model.evaluate()...")
    evaluate_start = time.time()
    resultados = modelo.evaluate(test_dataset, verbose=1)
    evaluate_time = time.time() - evaluate_start
    
    # Associar nomes de métricas aos resultados
    metric_names = ['loss'] + [m.name for m in modelo.metrics]
    resultados_evaluate = list(zip(metric_names, resultados))
    
    print(f"\n[{datetime.now().strftime('%H:%M:%S')}] Resultados da avaliação com model.evaluate() (tempo: {format_time(evaluate_time)}):")
    for name, value in resultados_evaluate:
        print(f"  - {name}: {value:.4f}")
    
    # Contar batches no dataset de teste
    test_batches = sum(1 for _ in test_dataset)
    print(f"\n[{datetime.now().strftime('%H:%M:%S')}] Dataset de teste contém {test_batches} batches")
    
    # Coletar predições e ground truths para análise detalhada
    print(f"\n[{datetime.now().strftime('%H:%M:%S')}] Iniciando coleta de predições para análise detalhada...")
    predict_start = time.time()
    
    y_true_all = []
    y_pred_all = []
    
    # Usar tqdm se disponível para mostrar uma barra de progresso, caso contrário, fazer manualmente
    try:
        from tqdm import tqdm
        # Usar tqdm para barra de progresso
        for i, (images, masks) in enumerate(tqdm(test_dataset, desc="Processando batches", unit="batch")):
            # Fazer predições
            batch_start = time.time()
            preds = modelo.predict(images, verbose=0)
            batch_time = time.time() - batch_start
            
            # Adicionar aos arrays completos
            y_true_all.append(masks.numpy())
            y_pred_all.append(preds)
            
            # Log a cada 5 batches ou no último batch
            if (i+1) % 5 == 0 or i+1 == test_batches:
                print(f"[{datetime.now().strftime('%H:%M:%S')}] Batch {i+1}/{test_batches} processado em {batch_time:.2f}s")
    except ImportError:
        # Sem tqdm, usar contador manual
        print(f"[{datetime.now().strftime('%H:%M:%S')}] Biblioteca tqdm não encontrada, usando contador manual")
        for i, (images, masks) in enumerate(test_dataset):
            # Mostrar progresso
            print(f"[{datetime.now().strftime('%H:%M:%S')}] Processando batch {i+1}/{test_batches}...", end='\r')
            
            # Fazer predições
            batch_start = time.time()
            preds = modelo.predict(images, verbose=0)
            batch_time = time.time() - batch_start
            
            # Adicionar aos arrays completos
            y_true_all.append(masks.numpy())
            y_pred_all.append(preds)
            
            # Log a cada 5 batches ou no último batch
            if (i+1) % 5 == 0 or i+1 == test_batches:
                print(f"[{datetime.now().strftime('%H:%M:%S')}] Batch {i+1}/{test_batches} processado em {batch_time:.2f}s")
    
    # Concatenar todos os batches
    print(f"[{datetime.now().strftime('%H:%M:%S')}] Concatenando resultados de todos os batches...")
    y_true_all = np.concatenate(y_true_all, axis=0)
    y_pred_all = np.concatenate(y_pred_all, axis=0)
    
    predict_time = time.time() - predict_start
    print(f"[{datetime.now().strftime('%H:%M:%S')}] Predições coletadas! Tempo: {format_time(predict_time)}")
    print(f"Formato das máscaras ground truth: {y_true_all.shape}")
    print(f"Formato das predições: {y_pred_all.shape}")
    
    # Calcular métricas detalhadas por classe
    print(f"\n[{datetime.now().strftime('%H:%M:%S')}] Calculando métricas detalhadas por classe...")
    metrics_start = time.time()
    
    # Binarizar predições com threshold 0.5
    print(f"[{datetime.now().strftime('%H:%M:%S')}] Binarizando predições (threshold=0.5)...")
    y_pred_bin = (y_pred_all > 0.5).astype(np.float32)
    
    # Inicializar arrays para métricas
    print(f"[{datetime.now().strftime('%H:%M:%S')}] Inicializando arrays para métricas...")
    iou_por_classe = np.zeros(NUM_CLASSES)
    precision_por_classe = np.zeros(NUM_CLASSES)
    recall_por_classe = np.zeros(NUM_CLASSES)
    f1_por_classe = np.zeros(NUM_CLASSES)
    support_por_classe = np.zeros(NUM_CLASSES)
    matriz_confusao = np.zeros((NUM_CLASSES, 2, 2))
    
    # Calculando métricas para cada classe
    print(f"[{datetime.now().strftime('%H:%M:%S')}] Calculando métricas para cada classe...")
    for i in range(NUM_CLASSES):
        # Mostrar progresso
        print(f"[{datetime.now().strftime('%H:%M:%S')}] Processando classe {i+1}/{NUM_CLASSES}...", end='\r')
        
        # Extrair máscaras para a classe atual
        y_true_class = y_true_all[..., i].flatten()
        y_pred_class = y_pred_bin[..., i].flatten()
        
        # Calcular IoU
        intersection = np.sum(y_true_class * y_pred_class)
        union = np.sum(y_true_class) + np.sum(y_pred_class) - intersection
        iou_por_classe[i] = intersection / (union + 1e-10)
        
        # Calcular precision, recall, f1
        true_positives = np.sum(y_true_class * y_pred_class)
        false_positives = np.sum((1 - y_true_class) * y_pred_class)
        false_negatives = np.sum(y_true_class * (1 - y_pred_class))
        true_negatives = np.sum((1 - y_true_class) * (1 - y_pred_class))
        
        precision_por_classe[i] = true_positives / (true_positives + false_positives + 1e-10)
        recall_por_classe[i] = true_positives / (true_positives + false_negatives + 1e-10)
        f1_por_classe[i] = 2 * precision_por_classe[i] * recall_por_classe[i] / (precision_por_classe[i] + recall_por_classe[i] + 1e-10)
        
        # Calcular support
        support_por_classe[i] = np.sum(y_true_class)
        
        # Preencher matriz de confusão
        matriz_confusao[i, 0, 0] = true_negatives
        matriz_confusao[i, 0, 1] = false_positives
        matriz_confusao[i, 1, 0] = false_negatives
        matriz_confusao[i, 1, 1] = true_positives
    
    print(f"[{datetime.now().strftime('%H:%M:%S')}] Calculando métricas globais...")
    # Calcular métricas globais (média ponderada)
    weight = support_por_classe / (np.sum(support_por_classe) + 1e-10)
    iou_medio = np.sum(iou_por_classe * weight)
    precision_global = np.sum(precision_por_classe * weight)
    recall_global = np.sum(recall_por_classe * weight)
    f1_global = np.sum(f1_por_classe * weight)
    
    # Organizar métricas em um dicionário
    metricas = {
        'iou_por_classe': iou_por_classe,
        'iou_medio': iou_medio,
        'precision_por_classe': precision_por_classe,
        'recall_por_classe': recall_por_classe,
        'f1_por_classe': f1_por_classe,
        'precision_global': precision_global,
        'recall_global': recall_global,
        'f1_global': f1_global,
        'matriz_confusao': matriz_confusao,
        'support': support_por_classe
    }
    
    metrics_time = time.time() - metrics_start
    print(f"[{datetime.now().strftime('%H:%M:%S')}] Métricas calculadas! Tempo: {format_time(metrics_time)}")
    
    # Verificar se atende aos critérios mínimos
    print(f"\n[{datetime.now().strftime('%H:%M:%S')}] ===== RESULTADOS DA AVALIAÇÃO DETALHADA =====")
    print(f"IoU Global Médio: {metricas['iou_medio']:.4f} - {'✅ APROVADO' if metricas['iou_medio'] >= 0.75 else '❌ REPROVADO'}")
    print(f"F1-score Global: {metricas['f1_global']:.4f} - {'✅ APROVADO' if metricas['f1_global'] >= 0.80 else '❌ REPROVADO'}")
    
    # Salvar resultados em tabelas
    print(f"\n[{datetime.now().strftime('%H:%M:%S')}] Salvando resultados em tabelas...")
    tables_start = time.time()
    
    # Criar DataFrame com métricas por classe
    df = pd.DataFrame({
        'Classe': CLASS_NAMES,
        'IoU': metricas['iou_por_classe'],
        'Precision': metricas['precision_por_classe'],
        'Recall': metricas['recall_por_classe'],
        'F1-score': metricas['f1_por_classe'],
        'Support (pixels)': metricas['support']
    })
    
    # Adicionar métricas globais
    df_global = pd.DataFrame({
        'Classe': ['Global'],
        'IoU': [metricas['iou_medio']],
        'Precision': [metricas['precision_global']],
        'Recall': [metricas['recall_global']],
        'F1-score': [metricas['f1_global']],
        'Support (pixels)': [np.sum(metricas['support'])]
    })
    
    # Concatenar DataFrames
    df_final = pd.concat([df, df_global], ignore_index=True)
    
    # Adicionar indicadores de status
    df_final['Status IoU'] = 'Aceitável'
    df_final.loc[df_final['IoU'] < 0.75, 'Status IoU'] = 'Abaixo do Mínimo'
    df_final.loc[df_final['IoU'] >= 0.85, 'Status IoU'] = 'Ideal'
    
    df_final['Status F1'] = 'Aceitável'
    df_final.loc[df_final['F1-score'] < 0.80, 'Status F1'] = 'Abaixo do Mínimo'
    df_final.loc[df_final['F1-score'] >= 0.85, 'Status F1'] = 'Ideal'
    
    # Salvar como CSV
    csv_path = os.path.join(RESULTS_DIR, 'resultados_metricas.csv')
    df_final.to_csv(csv_path, index=False)
    
    # Também salvar como formato LaTeX para o artigo
    latex_path = os.path.join(RESULTS_DIR, 'resultados_metricas.tex')
    with open(latex_path, 'w') as f:
        f.write(df_final.to_latex(index=False, float_format="%.4f"))
    
    # Tabela específica para classes desafiadoras
    df_desafiadoras = df.copy()
    df_desafiadoras = df_desafiadoras.sort_values(by='IoU', ascending=True)
    
    # Adicionar status
    df_desafiadoras['Status'] = 'Ideal (>0.85)'
    df_desafiadoras.loc[df_desafiadoras['IoU'] < 0.85, 'Status'] = 'Aceitável (0.75-0.85)'
    df_desafiadoras.loc[df_desafiadoras['IoU'] < 0.75, 'Status'] = 'Desafiadora (<0.75)'
    
    # Salvar tabela das classes desafiadoras
    desafiadoras_path = os.path.join(RESULTS_DIR, 'classes_desafiadoras.csv')
    df_desafiadoras.to_csv(desafiadoras_path, index=False)
    
    # Também salvar como formato LaTeX
    latex_desafiadoras_path = os.path.join(RESULTS_DIR, 'classes_desafiadoras.tex')
    with open(latex_desafiadoras_path, 'w') as f:
        f.write(df_desafiadoras.to_latex(index=False, float_format="%.4f"))
    
    tables_time = time.time() - tables_start
    print(f"[{datetime.now().strftime('%H:%M:%S')}] Tabelas salvas! Tempo: {format_time(tables_time)}")
    
    # Criar visualizações
    print(f"\n[{datetime.now().strftime('%H:%M:%S')}] Criando visualizações...")
    viz_start = time.time()
    
    # 1. Matriz de confusão global
    print(f"[{datetime.now().strftime('%H:%M:%S')}] Gerando matriz de confusão global...")
    plt.figure(figsize=(10, 8))
    
    # Calcular matriz de confusão global (soma de todas as classes)
    global_cm = np.sum(matriz_confusao, axis=0)
    
    # Normalizar para porcentagens
    global_cm_norm = global_cm / np.sum(global_cm) * 100
    
    sns.heatmap(global_cm_norm, annot=True, fmt='.1f', cmap='Blues',
                xticklabels=['Negativo', 'Positivo'],
                yticklabels=['Negativo', 'Positivo'])
    plt.ylabel('Classe Real')
    plt.xlabel('Classe Predita')
    plt.title('Matriz de Confusão Global (%)') 
    plt.tight_layout()
    plt.savefig(os.path.join(RESULTS_DIR, 'matriz_confusao_global.png'), dpi=300)
    plt.close()
    
    # 2. Matrizes de confusão por classe
    print(f"[{datetime.now().strftime('%H:%M:%S')}] Gerando matrizes de confusão por classe...")
    n_cols = 4
    n_rows = (NUM_CLASSES + n_cols - 1) // n_cols
    
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(20, 5*n_rows))
    axes = axes.flatten()
    
    for i in range(NUM_CLASSES):
        print(f"[{datetime.now().strftime('%H:%M:%S')}] Processando classe {i+1}/{NUM_CLASSES} para matriz de confusão...", end='\r')
        
        if i < len(axes):
            # Normalizar para porcentagens
            cm_i = matriz_confusao[i] / np.sum(matriz_confusao[i]) * 100
            
            sns.heatmap(cm_i, annot=True, fmt='.1f', cmap='Blues',
                        xticklabels=['Negativo', 'Positivo'],
                        yticklabels=['Negativo', 'Positivo'],
                        ax=axes[i])
            axes[i].set_title(f'Classe: {CLASS_NAMES[i]}')
            axes[i].set_ylabel('Real')
            axes[i].set_xlabel('Predito')
    
    # Esconder eixos em excesso
    for j in range(NUM_CLASSES, len(axes)):
        axes[j].axis('off')
    
    plt.tight_layout()
    plt.savefig(os.path.join(RESULTS_DIR, 'matriz_confusao_por_classe.png'), dpi=300)
    plt.close()
    
    # 3. Precision e Recall por classe
    print(f"[{datetime.now().strftime('%H:%M:%S')}] Gerando gráfico de Precision e Recall por classe...")
    plt.figure(figsize=(14, 6))
    
    # Criar barras agrupadas
    x = np.arange(NUM_CLASSES)
    width = 0.35
    
    plt.bar(x - width/2, precision_por_classe * 100, width, label='Precision (%)')
    plt.bar(x + width/2, recall_por_classe * 100, width, label='Recall (%)')
    
    plt.ylabel('Percentual (%)')
    plt.title('Precision e Recall por Classe')
    plt.xticks(x, CLASS_NAMES, rotation=45, ha='right')
    plt.legend()
    
    plt.tight_layout()
    plt.savefig(os.path.join(RESULTS_DIR, 'precision_recall_por_classe.png'), dpi=300)
    plt.close()
    
    # 4. IoU, Precision, Recall e F1 por classe
    print(f"[{datetime.now().strftime('%H:%M:%S')}] Gerando gráficos de métricas por classe...")
    fig, axs = plt.subplots(2, 2, figsize=(16, 12))
    
    # IoU por classe
    axs[0, 0].bar(range(len(CLASS_NAMES)), iou_por_classe)
    axs[0, 0].set_title('IoU por Classe')
    axs[0, 0].set_ylabel('IoU')
    axs[0, 0].set_xticks(range(len(CLASS_NAMES)))
    axs[0, 0].set_xticklabels(CLASS_NAMES, rotation=45, ha='right')
    axs[0, 0].axhline(y=0.75, color='r', linestyle='--', label='Min. Aceitável (0.75)')
    axs[0, 0].axhline(y=0.85, color='g', linestyle='--', label='Ideal (0.85)')
    axs[0, 0].legend()
    
    # Precision por classe
    axs[0, 1].bar(range(len(CLASS_NAMES)), precision_por_classe)
    axs[0, 1].set_title('Precision por Classe')
    axs[0, 1].set_ylabel('Precision')
    axs[0, 1].set_xticks(range(len(CLASS_NAMES)))
    axs[0, 1].set_xticklabels(CLASS_NAMES, rotation=45, ha='right')
    
    # Recall por classe
    axs[1, 0].bar(range(len(CLASS_NAMES)), recall_por_classe)
    axs[1, 0].set_title('Recall por Classe')
    axs[1, 0].set_ylabel('Recall')
    axs[1, 0].set_xticks(range(len(CLASS_NAMES)))
    axs[1, 0].set_xticklabels(CLASS_NAMES, rotation=45, ha='right')
    
    # F1-score por classe
    axs[1, 1].bar(range(len(CLASS_NAMES)), f1_por_classe)
    axs[1, 1].set_title('F1-score por Classe')
    axs[1, 1].set_ylabel('F1-score')
    axs[1, 1].set_xticks(range(len(CLASS_NAMES)))
    axs[1, 1].set_xticklabels(CLASS_NAMES, rotation=45, ha='right')
    axs[1, 1].axhline(y=0.80, color='r', linestyle='--', label='Min. Aceitável (0.80)')
    axs[1, 1].axhline(y=0.85, color='g', linestyle='--', label='Ideal (0.85)')
    axs[1, 1].legend()
    
    plt.tight_layout()
    plt.savefig(os.path.join(RESULTS_DIR, 'metricas_por_classe.png'), dpi=300)
    plt.close()
    
    # 5. Classes ordenadas por IoU
    print(f"[{datetime.now().strftime('%H:%M:%S')}] Gerando gráfico de classes desafiadoras...")
    # Ordenar classes por IoU (do menor para o maior)
    idx_sort = np.argsort(iou_por_classe)
    sorted_class_names = [CLASS_NAMES[i] for i in idx_sort]
    iou_sorted = iou_por_classe[idx_sort]
    
    # Criar figura para classes desafiadoras
    plt.figure(figsize=(14, 6))
    
    # Criar barras com cores diferentes baseadas no valor
    bars = plt.bar(range(len(CLASS_NAMES)), iou_sorted)
    
    # Colorir barras baseado no critério
    for i, bar in enumerate(bars):
        if iou_sorted[i] < 0.75:
            bar.set_color('red')  # Abaixo do mínimo aceitável
        elif iou_sorted[i] < 0.85:
            bar.set_color('orange')  # Aceitável mas não ideal
        else:
            bar.set_color('green')  # Ideal
    
    plt.axhline(y=0.75, color='r', linestyle='--', label='Min. Aceitável (0.75)')
    plt.axhline(y=0.85, color='g', linestyle='--', label='Ideal (0.85)')
    plt.title('Classes Ordenadas por IoU (Identificando Classes Desafiadoras)')
    plt.ylabel('IoU')
    plt.xticks(range(len(CLASS_NAMES)), sorted_class_names, rotation=45, ha='right')
    plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(RESULTS_DIR, 'classes_desafiadoras.png'), dpi=300)
    plt.close()
    
    viz_time = time.time() - viz_start
    print(f"[{datetime.now().strftime('%H:%M:%S')}] Visualizações criadas! Tempo: {format_time(viz_time)}")
    
    # Criar e salvar relatório de avaliação
    print(f"\n[{datetime.now().strftime('%H:%M:%S')}] Criando relatório de avaliação...")
    report_start = time.time()
    
    # Verificar conformidade com os critérios mínimos
    iou_criterio = "✅ APROVADO" if iou_medio >= 0.75 else "❌ REPROVADO"
    f1_criterio = "✅ APROVADO" if f1_global >= 0.80 else "❌ REPROVADO"
    
    # Status do IoU
    if iou_medio >= 0.85:
        iou_status = "IDEAL"
    elif iou_medio >= 0.75:
        iou_status = "ACEITÁVEL"
    else:
        iou_status = "ABAIXO DO MÍNIMO"
    
    # Status do F1-score
    if f1_global >= 0.85:
        f1_status = "IDEAL"
    elif f1_global >= 0.80:
        f1_status = "ACEITÁVEL"
    else:
        f1_status = "ABAIXO DO MÍNIMO"
    
    # Identificar classes mais desafiadoras (IoU < 0.75)
    indices_desafiadores = np.where(iou_por_classe < 0.75)[0]
    classes_desafiadoras = [CLASS_NAMES[i] for i in indices_desafiadores]
    
    if len(classes_desafiadoras) == 0:
        classes_desafiadoras_str = "Nenhuma classe apresentou IoU abaixo do mínimo aceitável (0.75)"
    else:
        classes_desafiadoras_str = ", ".join(classes_desafiadoras)
    
    # Construir relatório
    relatorio = f"""
======== RELATÓRIO DE AVALIAÇÃO DO MODELO DEEPLABV3+ ========
Data: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}
Modelo avaliado: {MODEL_PATH}

=== TEMPO DE EXECUÇÃO ===
Carregamento do modelo: {format_time(evaluate_time)}
Avaliação (model.evaluate): {format_time(evaluate_time)}
Coleta de predições: {format_time(predict_time)}
Cálculo de métricas: {format_time(metrics_time)}
Geração de tabelas: {format_time(tables_time)}
Criação de visualizações: {format_time(viz_time)}

=== MÉTRICAS GLOBAIS ===
IoU Global Médio: {iou_medio:.4f} [{iou_status}] - {iou_criterio}
F1-score Global: {f1_global:.4f} [{f1_status}] - {f1_criterio}
Precision Global: {precision_global:.4f}
Recall Global: {recall_global:.4f}

=== RESULTADOS DO MODEL.EVALUATE() ===
"""
    
    # Adicionar resultados do model.evaluate()
    for name, value in resultados_evaluate:
        relatorio += f"{name}: {value:.4f}\n"
    
    relatorio += f"""
=== CLASSES DESAFIADORAS (IoU < 0.75) ===
{classes_desafiadoras_str}

=== RESUMO DA AVALIAÇÃO ===
O modelo apresenta um IoU global médio de {iou_medio:.4f}, o que é {"considerado ideal (acima de 0.85)" if iou_medio >= 0.85 else "aceitável (acima de 0.75)" if iou_medio >= 0.75 else "abaixo do mínimo aceitável (0.75)"}.
O F1-score global é de {f1_global:.4f}, o que é {"considerado ideal (acima de 0.85)" if f1_global >= 0.85 else "aceitável (acima de 0.80)" if f1_global >= 0.80 else "abaixo do mínimo aceitável (0.80)"}.

=== RECOMENDAÇÕES ===
"""
    
    # Adicionar recomendações com base nos resultados
    if iou_medio < 0.75 or f1_global < 0.80:
        relatorio += """
1. O modelo não atende aos critérios mínimos de aceitação. Considere retreinar com ajustes nos hiperparâmetros.
2. Preste atenção especial às classes desafiadoras identificadas acima.
3. Aumente o número de amostras das classes problemáticas ou aplique técnicas de balanceamento de classes.
4. Experimente diferentes técnicas de aumento de dados para melhorar a generalização.
"""
    elif iou_medio < 0.85 or f1_global < 0.85:
        relatorio += """
1. O modelo atende aos critérios mínimos, mas pode ser melhorado para atingir o nível ideal.
2. Considere técnicas de fine-tuning para aprimorar o desempenho nas classes menos favorecidas.
3. Avalie se o balanceamento de classes no conjunto de treinamento pode ser otimizado.
"""
    else:
        relatorio += """
1. O modelo apresenta excelente desempenho, atendendo aos critérios ideais para IoU e F1-score.
2. Para futuras versões, considere otimizar o modelo para eficiência computacional mantendo o mesmo nível de desempenho.
3. Avalie se é possível reduzir o número de parâmetros ou o tempo de inferência sem comprometer a qualidade.
"""
    
    relatorio += f"""
=== DETALHAMENTO DO DATASET ===
Total de imagens avaliadas: {y_true_all.shape[0]}
Resolução das imagens: {IMG_SIZE[0]} x {IMG_SIZE[1]}
Número de classes: {NUM_CLASSES}

=== ARQUIVOS GERADOS ===
- Tabelas detalhadas: {os.path.join(RESULTS_DIR, 'resultados_metricas.csv')} e .tex
- Tabela de classes desafiadoras: {os.path.join(RESULTS_DIR, 'classes_desafiadoras.csv')} e .tex
- Matriz de confusão global: {os.path.join(RESULTS_DIR, 'matriz_confusao_global.png')}
- Matriz de confusão por classe: {os.path.join(RESULTS_DIR, 'matriz_confusao_por_classe.png')}
- Precision e Recall por classe: {os.path.join(RESULTS_DIR, 'precision_recall_por_classe.png')}
- Métricas por classe: {os.path.join(RESULTS_DIR, 'metricas_por_classe.png')}
- Classes ordenadas por dificuldade: {os.path.join(RESULTS_DIR, 'classes_desafiadoras.png')}

Este relatório e todos os arquivos gerados estão no diretório: {RESULTS_DIR}
=========================================================
"""
    
    # Salvar relatório
    report_path = os.path.join(RESULTS_DIR, "relatorio_avaliacao.txt")
    with open(report_path, 'w') as f:
        f.write(relatorio)
    
    report_time = time.time() - report_start
    print(f"[{datetime.now().strftime('%H:%M:%S')}] Relatório de avaliação criado! Tempo: {format_time(report_time)}")
    
    # Tempo total de execução
    total_time = time.time() - total_start_time
    print(f"\n[{datetime.now().strftime('%H:%M:%S')}] Avaliação concluída! Tempo total: {format_time(total_time)}")
    print(f"\nTodos os resultados foram salvos no diretório: {RESULTS_DIR}")
    print(f"Relatório de avaliação: {report_path}")

if __name__ == "__main__":
    try:
        avaliar_modelo()
        print(f"[{datetime.now().strftime('%H:%M:%S')}] Script finalizado com sucesso!")
    except Exception as e:
        print(f"[{datetime.now().strftime('%H:%M:%S')}] Erro durante a execução: {str(e)}")
        import traceback
        traceback.print_exc()


In [ ]:
# Segunda tentativa de carregar o modelo deeplabv3plus
import os
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import load_model # type: ignore
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, precision_recall_fscore_support
import pathlib
from dotenv import load_dotenv

# Definir a função de perda focal para carregar o modelo
def focal_loss(alpha=0.25, gamma=2.0):
    def loss(y_true, y_pred):
        y_pred = tf.keras.backend.clip(y_pred, 1e-7, 1 - 1e-7)
        focal_loss = -alpha * y_true * tf.keras.backend.pow(1 - y_pred, gamma) * tf.keras.backend.log(y_pred)
        focal_loss -= (1 - alpha) * (1 - y_true) * tf.keras.backend.pow(y_pred, gamma) * tf.keras.backend.log(1 - y_pred)
        return tf.keras.backend.mean(focal_loss)
    return loss

# Definição das métricas personalizadas
def iou_metric(y_true, y_pred):
    # Aplicar thresholding para obter previsões binárias
    y_pred = tf.cast(y_pred > 0.5, tf.float32)
    
    # Calcular interseção e união
    intersection = tf.reduce_sum(y_true * y_pred, axis=[1, 2, 3])
    union = tf.reduce_sum(y_true, axis=[1, 2, 3]) + tf.reduce_sum(y_pred, axis=[1, 2, 3]) - intersection
    
    # Calcular IoU e retornar média
    iou = tf.where(union > 0, intersection / union, tf.ones_like(intersection))
    return tf.reduce_mean(iou)

def precision_metric(y_true, y_pred):
    y_pred = tf.cast(y_pred > 0.5, tf.float32)
    true_positives = tf.reduce_sum(y_true * y_pred, axis=[1, 2, 3])
    predicted_positives = tf.reduce_sum(y_pred, axis=[1, 2, 3])
    precision = tf.where(predicted_positives > 0,
                        true_positives / predicted_positives,
                        tf.ones_like(true_positives))
    return tf.reduce_mean(precision)

def recall_metric(y_true, y_pred):
    y_pred = tf.cast(y_pred > 0.5, tf.float32)
    true_positives = tf.reduce_sum(y_true * y_pred, axis=[1, 2, 3])
    possible_positives = tf.reduce_sum(y_true, axis=[1, 2, 3])
    recall = tf.where(possible_positives > 0, true_positives / possible_positives, tf.ones_like(true_positives))
    return tf.reduce_mean(recall)

def f1_score_metric(y_true, y_pred):
    precision = precision_metric(y_true, y_pred)
    recall = recall_metric(y_true, y_pred)
    return 2 * ((precision * recall) / (precision + recall + tf.keras.backend.epsilon()))

def load_test_dataset():
    """
    Carrega o conjunto de dados de teste
    """
    # Carregar variáveis de ambiente
    load_dotenv()
    
    img_dir_str = os.getenv("BASE_IMG_FOLDER")
    if img_dir_str is None:
        raise ValueError("A variável de ambiente BASE_IMG_FOLDER não está definida no arquivo .env")
    img_dir = pathlib.Path(img_dir_str)
    
    mask_dir_str = os.getenv("NUMPY_FOLDER")
    if mask_dir_str is None:
        raise ValueError("A variável de ambiente NUMPY_FOLDER não está definida no arquivo .env")
    mask_dir = pathlib.Path(mask_dir_str)
    
    # Funções para extrair número do arquivo
    def get_file_number(filepath):
        return int(os.path.basename(filepath).replace('.png', ''))
    
    def get_file_number_mask(filepath):
        return int(os.path.basename(filepath).replace('.npy', ''))
    
    # Listar arquivos
    mask_files = [str(path) for path in mask_dir.glob('*.npy') if os.path.exists(path)]
    image_files = [str(path) for path in img_dir.glob('*.png') if os.path.exists(path)]
    
    # Ordenar
    image_files = sorted(image_files, key=get_file_number)
    mask_files = sorted(mask_files, key=get_file_number_mask)
    
    def load_and_preprocess_image_mask(image_path, mask_path):
        try:
            # Carregar imagem
            img = tf.io.read_file(image_path)
            img = tf.image.decode_png(img, channels=3)
            img = tf.image.resize(img, [512, 512])
            img = tf.cast(img, tf.float32) / 255.0
            
            # Carregar máscara
            mask = np.load(mask_path.numpy().decode())
            mask = tf.convert_to_tensor(mask, dtype=tf.float32)
            mask = mask / 255.0
            mask = tf.where(mask >= 0.3, 1.0, 0.0)
            
            return img, mask
        except Exception as e:
            print(f"Erro ao processar {image_path}, {mask_path}: {str(e)}")
            raise
    
    def process_path(image_path, mask_path):
        img, mask = tf.py_function(load_and_preprocess_image_mask, [image_path, mask_path], [tf.float32, tf.float32])
        img.set_shape((512, 512, 3))
        mask.set_shape((512, 512, 22))
        return img, mask
    
    # Criar dataset
    dataset = tf.data.Dataset.from_tensor_slices((image_files, mask_files))
    dataset = dataset.map(process_path, num_parallel_calls=tf.data.AUTOTUNE)
    
    # Dividir dataset
    dataset_size = tf.data.experimental.cardinality(dataset).numpy()
    train_size = int(0.8 * dataset_size)
    val_size = int(0.1 * dataset_size)
    
    # Dividir usando a mesma metodologia do treinamento
    shuffled_dataset = dataset.shuffle(buffer_size=32)
    train_dataset = shuffled_dataset.take(train_size)
    remaining_dataset = shuffled_dataset.skip(train_size)
    val_dataset = remaining_dataset.take(val_size)
    test_dataset = remaining_dataset.skip(val_size)
    
    # Configurar batch
    test_dataset = test_dataset.batch(8).prefetch(1)
    
    return test_dataset

# Função para extrair dados do dataset para avaliação por classe
def extract_dataset_data(dataset):
    images = []
    masks = []
    
    for img_batch, mask_batch in dataset:
        images.append(img_batch.numpy())
        masks.append(mask_batch.numpy())
    
    return np.vstack(images), np.vstack(masks)

# Função para calcular IoU por classe e gerar matriz de confusão
def evaluate_per_class(model, test_dataset, num_classes):
    print("Extraindo dados do dataset de teste...")
    test_images, test_masks = extract_dataset_data(test_dataset)
    
    print(f"Realizando previsões para {len(test_images)} imagens...")
    # Fazer previsões
    predictions = model.predict(test_images)
    
    # Converter para classes - para máscaras multiclasse
    pred_masks = np.argmax(predictions, axis=-1)
    true_masks = np.argmax(test_masks, axis=-1)
    
    # Inicializar arrays para armazenar IoU por classe
    class_iou = np.zeros(num_classes)
    
    # Calcular IoU para cada classe
    print("Calculando IoU por classe...")
    for class_id in range(num_classes):
        # Criar máscaras binárias para esta classe
        true_class = (true_masks == class_id).flatten()
        pred_class = (pred_masks == class_id).flatten()
        
        # Se não há pixels desta classe nas máscaras verdadeiras, pular
        if true_class.sum() == 0 and pred_class.sum() == 0:
            class_iou[class_id] = 1.0  # IoU perfeito se classe não está presente
            continue
        
        # Calcular interseção e união
        intersection = np.logical_and(true_class, pred_class).sum()
        union = np.logical_or(true_class, pred_class).sum()
        
        # Calcular IoU
        if union > 0:
            class_iou[class_id] = intersection / union
        else:
            class_iou[class_id] = 0.0  # Nenhuma sobreposição
    
    # Calcular métricas por classe usando sklearn
    print("Calculando precision, recall e F1-score por classe...")
    precision, recall, f1, support = precision_recall_fscore_support(
        true_masks.flatten(), pred_masks.flatten(), 
        labels=range(num_classes), average=None, zero_division=0
    )
    
    # Calcular matriz de confusão
    print("Gerando matriz de confusão...")
    cm = confusion_matrix(true_masks.flatten(), pred_masks.flatten(), labels=range(num_classes))
    
    # Normalizar a matriz de confusão (por linha)
    cm_normalized = np.zeros_like(cm, dtype=float)
    for i in range(cm.shape[0]):
        if cm[i].sum() > 0:
            cm_normalized[i] = cm[i] / cm[i].sum()
    
    return class_iou, precision, recall, f1, support, cm, cm_normalized

# Função principal para carregar modelo e avaliar
def evaluate_model(model_path, class_names):
    # Carregar dataset de teste
    print("Carregando dataset de teste...")
    test_dataset = load_test_dataset()
    
    # Carregar o modelo com as métricas personalizadas
    print(f"Carregando modelo de {model_path}...")
    custom_objects = {
        'loss': focal_loss(),
        'iou_metric': iou_metric,
        'precision_metric': precision_metric,
        'recall_metric': recall_metric,
        'f1_score_metric': f1_score_metric
    }
    
    model = load_model(model_path, custom_objects=custom_objects)
    
    # Avaliação básica do modelo
    print("Avaliando modelo no conjunto de teste...")
    results = model.evaluate(test_dataset)
    
    # Exibir resultados gerais
    print("\n--- Métricas Globais ---")
    for i, metric_name in enumerate(model.metrics_names):
        print(f"{metric_name}: {results[i]:.4f}")
    
    # Calcular métricas por classe
    num_classes = 22
    
    print(f"\nCalculando métricas detalhadas para {num_classes} classes...")
    class_iou, precision, recall, f1, support, cm, cm_normalized = evaluate_per_class(model, test_dataset, num_classes)
    
    # Exibir IoU por classe
    print("\n--- IoU por Classe ---")
    for i, class_name in enumerate(class_names):
        print(f"{class_name}: {class_iou[i]:.4f}")
    
    # Exibir métricas detalhadas por classe
    print("\n--- Métricas Detalhadas por Classe ---")
    print("Classe\tPrecision\tRecall\tF1-Score\tSupport")
    for i, class_name in enumerate(class_names):
        print(f"{class_name}\t{precision[i]:.4f}\t{recall[i]:.4f}\t{f1[i]:.4f}\t{support[i]}")
    
    # Calcular média global (ponderada pelo suporte)
    valid_indices = support > 0
    weighted_iou = np.sum(class_iou[valid_indices] * support[valid_indices]) / np.sum(support[valid_indices])
    weighted_precision = np.sum(precision[valid_indices] * support[valid_indices]) / np.sum(support[valid_indices])
    weighted_recall = np.sum(recall[valid_indices] * support[valid_indices]) / np.sum(support[valid_indices])
    weighted_f1 = np.sum(f1[valid_indices] * support[valid_indices]) / np.sum(support[valid_indices])
    
    print("\n--- Médias Globais (Ponderadas por suporte) ---")
    print(f"IoU Médio: {weighted_iou:.4f}")
    print(f"Precision Média: {weighted_precision:.4f}")
    print(f"Recall Médio: {weighted_recall:.4f}")
    print(f"F1-Score Médio: {weighted_f1:.4f}")
    
    # Também calcular média simples para comparação
    print("\n--- Médias Globais (Simples, apenas classes com suporte) ---")
    print(f"IoU Médio: {np.mean(class_iou[valid_indices]):.4f}")
    print(f"Precision Média: {np.mean(precision[valid_indices]):.4f}")
    print(f"Recall Médio: {np.mean(recall[valid_indices]):.4f}")
    print(f"F1-Score Médio: {np.mean(f1[valid_indices]):.4f}")
    
    # Visualizar matriz de confusão
    plt.figure(figsize=(14, 12))
    sns.heatmap(cm_normalized, annot=True, fmt='.2f', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names)
    plt.xlabel('Previsão')
    plt.ylabel('Verdadeiro')
    plt.title('Matriz de Confusão Normalizada')
    plt.tight_layout()
    plt.savefig('confusion_matrix.png', dpi=300)
    plt.show()
    
    # Visualizar IoU por classe em um gráfico de barras
    plt.figure(figsize=(14, 8))
    plt.bar(range(num_classes), class_iou)
    plt.xlabel('Classes')
    plt.ylabel('IoU')
    plt.title('IoU por Classe')
    plt.xticks(range(num_classes), class_names, rotation=90)
    plt.grid(axis='y', linestyle='--', alpha=0.7)
    for i, v in enumerate(class_iou):
        plt.text(i, v + 0.02, f'{v:.2f}', ha='center', va='bottom', rotation=90)
    plt.tight_layout()
    plt.savefig('iou_por_classe.png', dpi=300)
    plt.show()
    
    # Salvar resultados em um arquivo de texto
    with open('model_evaluation_results.txt', 'w') as f:
        f.write("--- Métricas Globais ---\n")
        for i, metric_name in enumerate(model.metrics_names):
            f.write(f"{metric_name}: {results[i]:.4f}\n")
        
        f.write("\n--- IoU por Classe ---\n")
        for i, class_name in enumerate(class_names):
            f.write(f"{class_name}: {class_iou[i]:.4f}\n")
        
        f.write("\n--- Métricas Detalhadas por Classe ---\n")
        f.write("Classe\tPrecision\tRecall\tF1-Score\tSupport\n")
        for i, class_name in enumerate(class_names):
            f.write(f"{class_name}\t{precision[i]:.4f}\t{recall[i]:.4f}\t{f1[i]:.4f}\t{support[i]}\n")
        
        f.write("\n--- Médias Globais (Ponderadas por suporte) ---\n")
        f.write(f"IoU Médio: {weighted_iou:.4f}\n")
        f.write(f"Precision Média: {weighted_precision:.4f}\n")
        f.write(f"Recall Médio: {weighted_recall:.4f}\n")
        f.write(f"F1-Score Médio: {weighted_f1:.4f}\n")
        
        f.write("\n--- Médias Globais (Simples, apenas classes com suporte) ---\n")
        f.write(f"IoU Médio: {np.mean(class_iou[valid_indices]):.4f}\n")
        f.write(f"Precision Média: {np.mean(precision[valid_indices]):.4f}\n")
        f.write(f"Recall Médio: {np.mean(recall[valid_indices]):.4f}\n")
        f.write(f"F1-Score Médio: {np.mean(f1[valid_indices]):.4f}\n")
    
    # Verificar se atende aos critérios mínimos
    print("\n--- Verificação dos Critérios Mínimos ---")
    print(f"IoU global médio: {weighted_iou:.4f} - {'APROVADO' if weighted_iou >= 0.75 else 'REPROVADO'} (mínimo: 0.75)")
    print(f"IoU global médio ideal: {weighted_iou:.4f} - {'APROVADO' if weighted_iou >= 0.85 else 'REPROVADO'} (ideal: 0.85)")
    print(f"F1-score global: {weighted_f1:.4f} - {'APROVADO' if weighted_f1 >= 0.80 else 'REPROVADO'} (mínimo: 0.80)")
    print(f"F1-score global ideal: {weighted_f1:.4f} - {'APROVADO' if weighted_f1 >= 0.85 else 'REPROVADO'} (ideal: 0.85)")
    
    return {
        'model': model,
        'global_metrics': results,
        'class_iou': class_iou,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'support': support,
        'confusion_matrix': cm,
        'confusion_matrix_normalized': cm_normalized
    }

# Exemplo de uso
if __name__ == "__main__":
    # Caminho para o modelo
    model_path = "deeplabv3plusF_2025-03-27_13-34-11.h5"
    
    # Defina os nomes das classes aqui (substitua pelos nomes corretos das suas 22 classes)
    class_names = [
        "Classe 0",  "Classe 1",  "Classe 2",  "Classe 3",  "Classe 4",
        "Classe 5",  "Classe 6",  "Classe 7",  "Classe 8",  "Classe 9",
        "Classe 10", "Classe 11", "Classe 12", "Classe 13", "Classe 14",
        "Classe 15", "Classe 16", "Classe 17", "Classe 18", "Classe 19",
        "Classe 20", "Classe 21"
    ]
    
    # Avalie o modelo
    evaluation = evaluate_model(model_path, class_names)

Carregando dataset de teste...
Carregando modelo de deeplabv3plusF_2025-03-27_13-34-11.h5...
